# Lab 4 — Naïve Bayes Classifier (Text / Spam)
**Dataset:** `spam.csv`  ·  `v1` = label (ham/spam), `v2` = message text  — **do not change**

**Objective**
- **Part 1:** implement **Multinomial Naive Bayes from scratch** (NumPy / Pandas only)
- **Part 2:** reproduce with **scikit-learn**, then compare on the same split

**Rules**
- Section 0 scaffolding is **GIVEN** — do not modify.
- Complete every `# TODO:` and replace each `___`.
- Both parts reuse the same `X_train / X_test / y_train / y_test` and the same `evaluate()` helper.

## Section 0 — Setup & Shared Pipeline  *(GIVEN — do not modify)*

In [1]:
dataset = "spam.csv"

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)

In [2]:
# Load (GIVEN)
data = pd.read_csv(dataset, encoding="latin-1")
print("shape:", data.shape)
print(data["v1"].value_counts())        # v1 = label, v2 = message text

shape: (5572, 5)
v1
ham     4825
spam     747
Name: count, dtype: int64


In [3]:
# Basic preprocessing (GIVEN)
data["v1"] = data["v1"].map({"spam": 1, "ham": 0})        # label -> binary (1 = spam)
vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(data["v2"])                   # X: [N, V] sparse count matrix
y = data["v1"].values                                      # y: [N]
print("vocab size V =", X.shape[1])

vocab size V = 8404


In [4]:
# Split (GIVEN) — keep original config
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

train: (3733, 8404) | test: (1839, 8404)


In [5]:
def evaluate(y_true, y_pred, y_prob=None, title=""):
    """Shared evaluation printer (GIVEN)."""
    print(f"=== {title} ===")
    print("confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("precision:", round(precision_score(y_true, y_pred), 4))
    print("recall   :", round(recall_score(y_true, y_pred), 4))
    print("f1       :", round(f1_score(y_true, y_pred), 4))
    if y_prob is not None:
        print("ROC AUC  :", round(roc_auc_score(y_true, y_prob), 4))

## Part 1 — Implement from scratch (NumPy / Pandas only)

Multinomial NB pipeline:
1. `fit` \u2192 log prior $\log P(c)$, Laplace-smoothed log-likelihood per word $\log P(w\mid c)$
2. `predict` \u2192 score $= \log P(c) + \sum_w x_w \log P(w\mid c)$, then MAP (argmax)

Smoothing: $P(w\mid c) = \dfrac{\text{count}(w,c) + \alpha}{\text{total}(c) + \alpha\,V}$

In [6]:
class MultinomialNBScratch:
    """Multinomial Naive Bayes — student implementation."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        # X: [N, V] sparse count matrix, y: [N]
        self.classes_ = np.unique(y)
        n_docs, V = X.shape
        self.log_prior_, self.log_likelihood_ = {}, {}
        for c in self.classes_:
            Xc = X[y == c]                       # Xc: [N_c, V]
            # TODO: log prior = log( N_c / N )
            self.log_prior_[c] = np.log(Xc.shape[0] / n_docs)
            # TODO: word counts per feature for class c  -> 1-D array shape [V]
            #       (sum Xc over documents; use np.asarray(...).ravel())
            word_counts = np.asarray(Xc.sum(axis=0)).ravel()
            # TODO: total word count in class c (scalar)
            total_count = word_counts.sum()
            # TODO: Laplace-smoothed log-likelihood vector -> shape [V]
            #       log( (word_counts + alpha) / (total_count + alpha * V) )
            self.log_likelihood_[c] = np.log((word_counts + self.alpha) / (total_count + self.alpha * V))
        return self

    def predict(self, X):
        # X: [M, V]  ->  y_pred: [M]
        scores = np.zeros((X.shape[0], len(self.classes_)))     # scores: [M, n_classes]
        for j, c in enumerate(self.classes_):
            # TODO: score for class c = X @ log_likelihood[c] + log_prior[c]
            #       (X is sparse; use np.asarray(X @ vec).ravel())
            scores[:, j] = np.asarray(X @ self.log_likelihood_[c]).ravel() + self.log_prior_[c]
        # TODO: MAP — index of the max score per row, then map back to class label
        best_idx = np.argmax(scores, axis=1)
        return self.classes_[best_idx]

In [7]:
model = MultinomialNBScratch(alpha=1.0)
model.fit(X_train, y_train)            # TODO: fit model on the training set
y_pred_scratch = model.predict(X_test) # TODO: predict on X_test
evaluate(y_test, y_pred_scratch, None, "Part 1 - MultinomialNB (from scratch)")

=== Part 1 - MultinomialNB (from scratch) ===
confusion matrix:
 [[1563   24]
 [  20  232]]
accuracy : 0.9761
precision: 0.9062
recall   : 0.9206
f1       : 0.9134


## Part 2 — Using scikit-learn

Reproduce the classifier with `sklearn.naive_bayes.MultinomialNB` on the **same split**.

In [8]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB(alpha=1.0)            # TODO: instantiate MultinomialNB(alpha=1.0)
clf.fit(X_train, y_train)                 # TODO: fit clf on (X_train, y_train)
y_pred_sk = clf.predict(X_test)           # TODO: predict labels on X_test
y_prob_sk = clf.predict_proba(X_test)[:, 1]  # TODO: predict P(spam) on X_test  (predict_proba, column 1)
evaluate(y_test, y_pred_sk, y_prob_sk, "Part 2 - sklearn MultinomialNB")

=== Part 2 - sklearn MultinomialNB ===
confusion matrix:
 [[1563   24]
 [  20  232]]
accuracy : 0.9761
precision: 0.9062
recall   : 0.9206
f1       : 0.9134
ROC AUC  : 0.9822


## Section 3 — Compare

| Metric | Part 1 (scratch) | Part 2 (sklearn) |
|---|---|---|
| Accuracy | 0.9761 | 0.9761 |
| Precision | 0.9062 | 0.9062 |
| Recall | 0.9206 | 0.9206 |
| F1 | 0.9134 | 0.9134 |

**Q:** Với spam filter, giữa **precision** và **recall** metric nào ưu tiên hơn? Tại sao? →

Với bộ lọc spam thì precision quan trọng hơn. Nếu mô hình chặn nhầm một email thật, email đó bị đưa vào thư mục rác và người dùng có thể mất luôn thông tin quan trọng mà không hề biết. Ngược lại nếu mô hình bỏ lọt một email spam vào hộp thư đến thì người dùng chỉ hơi phiền, xóa đi là xong.

Precision cho biết trong số email bị đánh dấu là spam thì bao nhiêu phần thực sự là spam, nên precision cao đồng nghĩa với việc email thật ít bị chặn oan. Recall vẫn cần để lọc được nhiều spam, nhưng chỉ là mục tiêu phụ: nên giữ precision cao trước rồi mới tìm cách nâng recall.